In [0]:
%sql
USE CATALOG shopsphere;

CREATE TABLE IF NOT EXISTS shopsphere.silver.inventory(
   inventory_date DATE,
    product_id INT,
    warehouse_id STRING,
    available_quantity INT,
    reserved_quantity INT,
    reorder_level INT
);

CREATE TABLE IF NOT EXISTS shopsphere.quarantine.inventory(
    inventory_date DATE,
    product_id INT,
    warehouse_id STRING,
    available_quantity INT,
    reserved_quantity INT,
    reorder_level INT
)
USING DELTA;

In [0]:
from pyspark.sql.functions import *
from delta.tables import *

df = spark.read.table("shopsphere.bronze.inventory")

In [0]:
availability = col('available_quantity')-col('reserved_quantity')

df_valid = df.withColumn("available_to_sell",
                         when(availability<=0,lit("OUT OF STOCK"))
                         .when(availability<col('reorder_level'),lit("LOW STOCK"))
                         .otherwise(lit("IN STOCK")))

In [0]:
#write transformed data to silver table
silver_table = DeltaTable.forName(spark,"shopsphere.silver.inventory")

silver_table.alias("target").merge(df_valid.alias("source"),
                                       "target.product_id = source.product_id").\
                                        whenNotMatchedInsertAll().\
                                        withSchemaEvolution().\
                                        execute()